# 📊 KPI Monitoring System — Exploratory Analysis

**Project:** Automated Business KPI Monitoring & Reporting System  
**Author:** Business Analyst  
**Date:** 2024

This notebook performs EDA across the five business tables and derives key insights.

---

## Sections
1. Setup & data loading
2. Call analysis
3. Campaign analysis
4. Agent performance
5. Customer segment analysis
6. Financial KPIs
7. Business insights & recommendations

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'etl'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import sqlalchemy as sa
import warnings
warnings.filterwarnings('ignore')

from load import get_engine
engine = get_engine()

def sql(q):
    with engine.connect() as conn:
        return pd.read_sql(sa.text(q), conn)

print('✅ Connected to database')

## 1. Dataset Overview

In [ ]:
for tbl in ['agents','campaigns','leads','calls','sales']:
    n = sql(f'SELECT COUNT(*) AS n FROM {tbl}').iloc[0]['n']
    print(f'{tbl:<12}: {n:>7,} rows')

## 2. Call Analysis — Connection Rate Trend

In [ ]:
daily = sql("""
    SELECT call_date,
           COUNT(*) AS total_calls,
           SUM(is_connected) AS connected_calls,
           ROUND(100.0 * SUM(is_connected) / COUNT(*), 2) AS conn_rate
    FROM calls
    GROUP BY call_date
    ORDER BY call_date
""")

fig = px.line(daily, x='call_date', y='conn_rate',
              title='Daily Connection Rate — Note the dip after August 2024',
              labels={'call_date':'Date','conn_rate':'Connection Rate (%)'})
fig.add_hline(y=50, line_dash='dash', line_color='red', annotation_text='50% Warning Threshold')
fig.show()

# Key stats
print(f'Overall avg connection rate: {daily.conn_rate.mean():.2f}%')
print(f'Lowest daily rate: {daily.conn_rate.min():.2f}% on {daily.loc[daily.conn_rate.idxmin(),"call_date"]}')
print(f'Post-August avg: {daily[daily.call_date >= "2024-08-01"].conn_rate.mean():.2f}%')

## 3. Agent Performance — Anomaly Detection (Agent 5)

In [ ]:
agents = sql("""
    SELECT a.agent_name, a.team,
           COUNT(*) AS total_calls,
           SUM(is_connected) AS connected,
           ROUND(100.0 * SUM(is_connected) / COUNT(*), 2) AS conn_rate,
           COALESCE(SUM(s.revenue), 0) AS revenue
    FROM agents a
    JOIN calls c ON a.agent_id = c.agent_id
    LEFT JOIN sales s ON c.lead_id = s.lead_id
    GROUP BY a.agent_id, a.agent_name, a.team
    ORDER BY conn_rate
""")

print('Bottom 5 agents by connection rate:')
print(agents.head(5)[['agent_name','team','conn_rate','revenue']].to_string(index=False))

fig = px.scatter(agents, x='total_calls', y='conn_rate', color='team',
                 size='revenue', hover_name='agent_name',
                 title='Agent Performance: Calls vs Connection Rate (bubble = revenue)')
fig.add_hline(y=50, line_dash='dash', line_color='red')
fig.show()

## 4. Campaign Analysis — CAC Anomaly (Campaign 3: Summer Blitz)

In [ ]:
campaigns = sql("""
    SELECT c.campaign_name,
           COUNT(s.sale_id) AS total_sales,
           ROUND(SUM(s.revenue), 2) AS revenue,
           ROUND(AVG(s.acquisition_cost), 2) AS avg_cac,
           ROUND(SUM(s.revenue) / NULLIF(SUM(s.acquisition_cost), 0), 2) AS roas
    FROM sales s
    JOIN campaigns c ON s.campaign_id = c.campaign_id
    GROUP BY c.campaign_id, c.campaign_name
    ORDER BY avg_cac DESC
""")

print(campaigns.to_string(index=False))

fig = px.bar(campaigns, x='campaign_name', y='avg_cac',
             color='avg_cac', color_continuous_scale='RdYlGn_r',
             title='Average CAC by Campaign — Summer Blitz should stand out')
fig.add_hline(y=400, line_dash='dash', line_color='orange', annotation_text='Warning ₹400')
fig.add_hline(y=500, line_dash='dash', line_color='red',    annotation_text='Critical ₹500')
fig.show()

## 5. Financial KPIs — ROAS and Revenue per Campaign

In [ ]:
fin = sql("""
    SELECT c.campaign_name, c.budget,
           ROUND(SUM(s.revenue), 2) AS revenue,
           ROUND(SUM(s.revenue) / NULLIF(c.budget, 0), 2) AS roas
    FROM campaigns c
    LEFT JOIN sales s ON c.campaign_id = s.campaign_id
    GROUP BY c.campaign_id, c.campaign_name, c.budget
    ORDER BY roas DESC
""")

fig = go.Figure()
fig.add_trace(go.Bar(x=fin.campaign_name, y=fin.budget,  name='Budget',  marker_color='#94a3b8'))
fig.add_trace(go.Bar(x=fin.campaign_name, y=fin.revenue, name='Revenue', marker_color='#3985D6'))
fig.update_layout(barmode='group', title='Budget vs Revenue by Campaign',
                  xaxis_title='Campaign', yaxis_title='Amount (₹)')
fig.show()

## 6. Business Insights & Recommendations

| Finding | Evidence | Business Recommendation |
|---|---|---|
| Summer Blitz (Campaign 3) has abnormally high CAC | Avg CAC ≫ ₹400 | Review targeting, reduce ad spend, refine audience |
| Agent 5 has significantly lower connection rate | ~30% vs 55% avg | Coaching, call time adjustment, lead quality review |
| Connection rate dropped post-August 2024 | Daily trend dip | Investigate lead list quality and calling schedule |
| Enterprise segment generates highest revenue per sale | Avg revenue ≫ others | Prioritise Enterprise leads in future campaigns |
| Evening hours (6–8 PM) show better connection | Hour-of-day analysis | Increase call allocation in evening slots |

### Questions answered:
1. **What happened?** Connection rate declined from Aug 2024, one campaign has high CAC, one agent underperforms.
2. **Why might it have happened?** Possible lead quality degradation, wrong call timing, or poor targeting.
3. **What should the business do next?** Prioritise Enterprise segment, retrain Agent 5, audit Summer Blitz campaign spend.